# Education experiment review
This notebook explains immutable pipeline results; it does not duplicate training or metric logic.
First run the CLI in `docs/education-experiments.md`. Select a report using `EDUCATION_EXPERIMENT_REPORT`.
The dataset has already been explored: a new split is not external validation. Scores are not diagnoses.

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd().resolve()
while not (root / 'packages/ml/src').is_dir() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / 'packages/ml/src'))
import pandas as pd
import matplotlib.pyplot as plt
from churn_ml.infrastructure.filesystem.experiment_reports import read_report
directory = Path(os.environ.get('EDUCATION_EXPERIMENT_REPORT', str(root / 'artifacts/experiments/education-v1')))
report = read_report(directory.resolve())

## Evaluation protocol
Training fits preprocessing and models. Validation chooses the model and threshold. Final test is used only after selection.
Do not repeatedly tune against test results. Preserve this report and use new IDs for subsequent experiments.

In [ ]:
display(pd.DataFrame(report.get('splits', {})).T)
display(pd.Series(report.get('selection', {}), name='Frozen validation decision'))
display(pd.Series(report['evaluation']['metrics'], name='Final evaluation metrics'))
for warning in report.get('warnings', []):
    print(warning)

## Calibration and review workload
The diagonal represents agreement between average predicted scores and observed group rates. Sparse bins are uncertain; this figure does not establish clinical validity.
False positives are students flagged despite a Low/Medium recorded outcome. No operational review budget has been chosen.

In [ ]:
checks = report['evaluation']['diagnostics']
display(pd.Series(checks['confusion'], name='Confusion counts'))
display(pd.Series({key: checks[key] for key in ['brier_score', 'log_loss', 'false_positive_rate', 'flagged_count', 'flagged_fraction']}))
reliability = pd.DataFrame(checks['reliability'])
display(reliability)
ax = reliability.dropna().plot(x='mean_score', y='observed_rate', marker='o', label='Observed groups')
ax.plot([0, 1], [0, 1], '--', label='Perfect agreement')
ax.set(xlabel='Mean model score', ylabel='Observed High fraction', xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.show()

## Compare experiments
Use the local MLflow UI documented in the guide to compare test metrics, configurations and aggregate artifacts. Comparisons require the same dataset and split hashes.
The local tracker records the model and aggregate report. To retain this notebook as an artifact, log a reviewed copy explicitly; never log executed cells containing student records or credentials.
There is no automatically approved winner or student intervention.